# Build raw production train/test splits from Encoded.csv

This notebook:
- Loads `Encoded.csv`
- Creates a **train set (160,000 rows)**
- Creates a **test set (243,177 rows)**
- Preserves attack-type distribution (stratified)
- Ensures all classes are present

Outputs (default):
- `Encoded_raw_train_160000.csv`
- `Endoced_raw_test_20pct.csv` 

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_PATH = Path("Encoded.csv")

ATTACK_TYPE_COL = "Attack Type"

TRAIN_SIZE = 160_000
TEST_SIZE = 243_177

TRAIN_OUT = Path("Datasets/Encoded_train_raw_160000.csv")
TEST_OUT = Path("Datasets/Encoded_test_raw_243177.csv")

RANDOM_SEED = 42

def detect_attack_col(df, preferred):
    if preferred in df.columns:
        return preferred
    cols_lower = {c.lower(): c for c in df.columns}
    alt = cols_lower.get(preferred.lower())
    if alt:
        return alt
    raise ValueError("Attack type column not found")

def allocate_counts(counts, n):
    counts = counts.sort_index()
    total = counts.sum()
    raw = counts / total * n
    base = np.floor(raw).astype(int)
    base[base == 0] = 1

    diff = n - base.sum()

    if diff > 0:
        for cls in (raw - np.floor(raw)).sort_values(ascending=False).index:
            if diff == 0:
                break
            base[cls] += 1
            diff -= 1
    elif diff < 0:
        diff = -diff
        for cls in (raw - np.floor(raw)).sort_values().index:
            if diff == 0:
                break
            if base[cls] > 1:
                base[cls] -= 1
                diff -= 1

    return base

def stratified_sample_indices(df, col, n, seed):
    counts = df[col].value_counts()
    alloc = allocate_counts(counts, n)

    rng = np.random.default_rng(seed)
    idx_list = []

    for cls, k in alloc.items():
        cls_idx = df.index[df[col] == cls].to_numpy()
        idx_list.append(rng.choice(cls_idx, size=k, replace=False))

    idx = np.concatenate(idx_list)
    rng.shuffle(idx)
    return idx

df = pd.read_csv(INPUT_PATH)

attack_col = detect_attack_col(df, ATTACK_TYPE_COL)

train_idx = stratified_sample_indices(df, attack_col, TRAIN_SIZE, RANDOM_SEED)
train_df = df.loc[train_idx].copy()

remaining_df = df.drop(train_idx)

test_idx = stratified_sample_indices(remaining_df, attack_col, TEST_SIZE, RANDOM_SEED + 1)
test_df = remaining_df.loc[test_idx].copy()

TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)

train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (160000, 96)
Test: (243177, 96)
